# gVXR CT Simulator — Google Colab
Run these cells from top to bottom. Python keeps the gVXR simulation engine; the native C++ server handles the browser interface and HTTP API.

Upload `ct_core.py`, `ct_variations.py`, `simulator_engine.py`, `index.html`, `cpp_server.cpp`, `make_dataset.py`, `ct_voxel.cpp`, and `build_voxel.py`. The voxelizer extension is needed when generating ground-truth volumes; section 3 builds it with `build_voxel.py`.

## 1. Install system libraries and Python packages

The server embeds the same Python 3.13 interpreter used by this Colab kernel. Its C++ headers and library must be available for the matching interpreter; the build cell checks this before compiling.

In [ ]:
import subprocess, sys
!apt-get update -qq
!apt-get install -y -qq build-essential libgomp1 libxcb-res0 libxcb-xinerama0 libxcb-xkb1 libxcb-icccm4 libxcb-image0 \
  libxcb-keysyms1 libxcb-render-util0 libxcb-shape0 libxcb-randr0 libxkbcommon-x11-0 \
  libxrandr2 libxinerama1 libxcursor1 libxi6 libxxf86vm1 \
  libgl1 libegl1 libopengl0 libglu1-mesa libgl1-mesa-dri libegl-mesa0 libglx-mesa0 \
  libxcb-composite0 libxcb-cursor0 libxcb-dri2-0 libxcb-ewmh2 \
  libxcb-dri3-0 libxcb-glx0 libxcb-present0 libxcb-sync1 libxcb-xfixes0 libxcb-xinput0 \
  libxcb-util1 libxcb-xv0 libxcb-damage0 libxcb-dpms0 libxcb-record0 libxcb-screensaver0 \
  libxcb-shm0 libxcb-xtest0 libxcb-xf86dri0 libxcb-xvmc0
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gvxr', 'numpy', 'matplotlib', 'tifffile', 'trimesh', 'pycollada', 'pillow', 'scikit-image', 'scipy', 'h5py', 'pyyaml', 'xraylib', 'pybind11', 'setuptools'], check=True)

## 2. Add the project source files
Choose the Python sources, `index.html`, `cpp_server.cpp`, `ct_voxel.cpp`, and `build_voxel.py` in the upload dialog. If you already cloned or mounted the project in this runtime, skip the upload cell and set `PROJECT_DIR` to that folder in the next cell.

In [ ]:
from google.colab import files
import subprocess
uploaded = files.upload()  # ct_core.py, ct_variations.py, simulator_engine.py, index.html, cpp_server.cpp, make_dataset.py, ct_voxel.cpp, build_voxel.py
PROJECT_DIR = '/content' 


In [ ]:
import os
from pathlib import Path
os.chdir(PROJECT_DIR)
required = ['ct_core.py', 'ct_variations.py', 'simulator_engine.py', 'index.html', 'cpp_server.cpp', 'make_dataset.py', 'ct_voxel.cpp', 'build_voxel.py']
missing = [name for name in required if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Missing project files: ' + ', '.join(missing))
engine_source = Path('simulator_engine.py').read_text()
required_engine_helpers = ['def image_bytes(', 'def projection_bytes(', 'def thumb_bytes(', 'def batch_rows_json(']
missing_helpers = [name for name in required_engine_helpers if name not in engine_source]
if missing_helpers:
    raise RuntimeError('Upload the updated simulator_engine.py; missing web image helpers: ' + ', '.join(missing_helpers))
print('Project directory:', Path.cwd())

## 3. Build the C++ voxelizer
This builds the `ct_voxel` Python extension from `ct_voxel.cpp` using the project helper. It is needed when generating ground-truth volumes. The build uses the active Colab Python interpreter and its installed `pybind11` package.

In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable, 'build_voxel.py'], check=True)
importlib.invalidate_caches()
import ct_voxel
print('C++ voxelizer ready:', ct_voxel.__name__)

## 4. Start the web app

Build and start the native C++ HTTP server, then open it with one of the access methods below. If Colab's built-in proxy stays blank, use the Cloudflare tunnel in section 5A.

In [ ]:
import os, shlex, subprocess, sys, sysconfig, time
# Stop the previous server if this cell is rerun.
previous_server = globals().get('server')
if previous_server is not None and previous_server.poll() is None:
    previous_server.terminate()
    try:
        previous_server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_server.kill()
        previous_server.wait()
# Use headers and libraries for the active Python runtime.
include_dir = sysconfig.get_path('include')
libdir = sysconfig.get_config_var('LIBDIR')
ldlibrary = sysconfig.get_config_var('LDLIBRARY')
libfile = os.path.join(libdir or '', ldlibrary or '')
if not os.path.isfile(os.path.join(include_dir, 'Python.h')) or not os.path.isfile(libfile):
    raise RuntimeError(f'Python {sys.version_info.major}.{sys.version_info.minor} embed headers/library are missing. Need Python.h and libpython for {sys.executable}; the system python3-dev package may target a different Python.')
includes = ['-I' + include_dir]
link_args = [libfile, '-Wl,-rpath,' + libdir, *shlex.split(sysconfig.get_config_var('LIBS') or ''), *shlex.split(sysconfig.get_config_var('SYSLIBS') or '')]
subprocess.run(['c++', '-std=c++17', '-O2', '-pthread', *includes,
                'cpp_server.cpp', *link_args, '-o', 'ct_simulator_server'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
env['CT_SIMULATOR_LOCAL'] = '0'  # Colab uses its headless EGL context
env['CT_PYTHON_SITE'] = sysconfig.get_paths()['purelib']
server_log_path = os.path.join(PROJECT_DIR, 'ct_simulator_server.log')
server_log = open(server_log_path, 'w')
server = subprocess.Popen(['./ct_simulator_server'], cwd=PROJECT_DIR, env=env,
                          stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(3)
if server.poll() is not None:
    server_log.flush()
    server_log.close()
    print(Path(server_log_path).read_text(errors='replace'))
    raise RuntimeError(f'C++ server exited during startup (exit {server.returncode}); see log above.')
server_log.flush()
print('C++ server is running on port 5000')

## 5A. Open through a Cloudflare tunnel (temporary public URL)

Use this temporary URL if Colab's built-in proxy in section 5B stays blank. The tunnel connects to 127.0.0.1 so it reaches the server over IPv4. If the URL returns HTTP 502, check that the server cell is still running and restart this tunnel cell to get a fresh URL.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re, subprocess, time
previous_tunnel = globals().get('tunnel')
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill()
        previous_tunnel.wait()
previous_log = globals().get('cf_log')
if previous_log is not None and not previous_log.closed:
    previous_log.close()
cf_log = open('cloudflared.log', 'w')
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'], stdout=cf_log, stderr=cf_log)
url = None
for _ in range(40):
    time.sleep(1)
    log_text = open('cloudflared.log').read()
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log_text)
    if match:
        url = match.group(0)
        break
print('Open this temporary public URL:', url) if url else print(log_text[-2000:])

## 5B. Open through Colab's built-in proxy

This embeds the app in the notebook output. If the iframe stays blank even though the local server returns HTTP 200, use the Cloudflare tunnel in section 5A instead. Colab's `serve_kernel_port_as_window` link helper is deprecated, so use the iframe helper below.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(5000, height=900)

Choose one access method. Keep the runtime running while using the app. Stop the C++ server with `server.terminate()` and the Cloudflare tunnel with `tunnel.terminate()`. Run dataset generation from a code cell with `subprocess.run([sys.executable, 'make_dataset.py', 'CONFIG.yaml'], check=True)`.